# Naive BC on HumanoidMaze Medium

In [1]:
import random
import torch
import pickle
import os
import numpy as np
import matplotlib.pyplot as plt

from collections import defaultdict

from causal_gym import HumanoidMazePCH
from causal_rl.algo.imitation.imitate import *
from causal_rl.algo.imitation.finetune import *

<frozen importlib._bootstrap>:241: RuntimeWarning: Your system is avx2 capable but pygame was not built with support for it. The performance of some of your blits could be adversely affected. Consider enabling compile time detection with environment variables like PYGAME_DETECT_AVX2=1 if you are compiling without cross compilation.
/home/et2842/miniconda3/envs/causalenv/lib/python3.11/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


In [2]:
os.environ['CUDA_VISIBLE_DEVICES'] = '1'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [3]:
num_steps = 2000
seed = 0
lookback = 1
hidden_dims = {'V'}

random.seed(seed)
torch.manual_seed(seed)

In [4]:
# for training: regular W, O hidden
train_env = HumanoidMazePCH(num_steps=num_steps, expert_mode=True, custom_hidden=hidden_dims, seed=seed)

# for eval: corrupted W, O hidden
eval_env = HumanoidMazePCH(num_steps=num_steps, expert_mode=False, seed=seed)

## Causal Graph Analysis

In [5]:
# to save time; conceptually the same
small_steps = lookback + 1
small_env = HumanoidMazePCH(num_steps=small_steps, seed=seed)
G = parse_graph(small_env.get_graph)
X_small = {f'X{t}' for t in range(small_steps)}
Y = f'Y{small_steps}'

X = {f'X{t}' for t in range(num_steps)}
obs_prefix = train_env.env.observed_unobserved_vars[0]

In [6]:
naive_Z_sets = {}
for Xi in X:
    i = int(Xi[1:])
    cond = set()

    for j in range(i+1):
        cond.update({f'{o}{j}' for o in list(set(obs_prefix) - {'X'})})

    for j in range(i):
        cond.add(f'X{j}')
    naive_Z_sets[Xi] = cond

naive_Z_sets['X1']

{'A0',
 'A1',
 'C0',
 'C1',
 'E0',
 'E1',
 'H0',
 'H1',
 'J0',
 'J1',
 'P0',
 'P1',
 'W0',
 'W1',
 'X0'}

## Expert Trajectories

In [7]:
# for eval: corrupted W, O shown
traj_env = HumanoidMazePCH(num_steps=num_steps, expert_mode=True)
# load model
MODEL_PATH = '/home/et2842/causal/causalrl/models/humanoidmaze_medium_expert_finetuned.pt'
ckpt = torch.load(MODEL_PATH, map_location=device, weights_only=False)

action_bounds = (ckpt['action_bounds_low'], ckpt['action_bounds_high'])

expert_model = ContinuousPolicyNN(
    input_dim=ckpt['input_dim'],
    action_dim=ckpt['num_actions'],
    hidden_dim=256,
    num_blocks=ckpt['num_blocks'],
    dropout=ckpt['dropout'],
    layernorm=ckpt['layernorm'],
    final_tanh=ckpt['final_tanh'],
    action_bounds=action_bounds,
).to(device)

expert_model.load_state_dict(ckpt['state_dict'])
expert_model.eval()

slots = ckpt['slots']
Z_trim = ckpt['Z_trim']
dims = ckpt['dims']
lookback = ckpt['lookback']

expert_policy = shared_policy_fn_long_horizon(expert_model, slots, Z_trim, continuous=True, device=device)
expert_policies = make_shared_policy_dict(expert_policy)
num_eval_eps = 500

records = collect_imitator_trajectories(
    env=traj_env,
    policies=expert_policies,
    num_episodes=num_eval_eps,
    max_steps=num_steps,
    show_progress=True
)

len(records)

Starting episode 1/500...


  Episode 1 ended at step 2000 (terminated: False, truncated: True).
Starting episode 2/500...


  Episode 2 ended at step 2000 (terminated: False, truncated: True).
Starting episode 3/500...


  Episode 3 ended at step 2000 (terminated: False, truncated: True).
Starting episode 4/500...


  Episode 4 ended at step 2000 (terminated: False, truncated: True).
Starting episode 5/500...


...

  Episode 497 ended at step 2000 (terminated: False, truncated: True).
Starting episode 498/500...


  Episode 498 ended at step 2000 (terminated: False, truncated: True).
Starting episode 499/500...


  Episode 499 ended at step 2000 (terminated: False, truncated: True).
Starting episode 500/500...


  Episode 500 ended at step 1320 (terminated: True, truncated: False).
Finished collecting imitator trajectories.


975945

In [8]:
dims = {
    'P': 2,
    'A': 21,
    'H': 1,
    'E': 12,
    # 'V': 3,
    'C': 3,
    'J': 27,
    'W': 2,
    'X': 21
}

## Training

In [9]:
hidden_size = 256
lr = 3e-4
batch_size = 2048
patience = 15
num_blocks = 4
epochs = 200
dropout = 0.0

In [10]:
nbc_model, nbc_slots, nbc_Z_trim = train_single_policy_long_horizon(
    records,
    naive_Z_sets,
    dims=dims,
    epochs=epochs,
    include_vars=obs_prefix,
    lookback=lookback,
    continuous=True,
    num_actions=train_env.action_space.shape[0],
    hidden_dim=hidden_size,
    num_blocks=num_blocks,
    dropout=dropout,
    lr=lr,
    batch_size=batch_size,
    patience=patience,
    device=device,
    seed=seed,
    action_bounds=(train_env.action_space.low, train_env.action_space.high)
)

nbc_policy = shared_policy_fn_long_horizon(nbc_model, nbc_slots, nbc_Z_trim, continuous=True, device=device)
nbc_policies = make_shared_policy_dict(nbc_policy)

[LongHorizon] Epoch 1: train loss = 0.040414, val loss = 0.019276.


[LongHorizon] Epoch 2: train loss = 0.016665, val loss = 0.015069.


[LongHorizon] Epoch 3: train loss = 0.014067, val loss = 0.013541.


[LongHorizon] Epoch 4: train loss = 0.012778, val loss = 0.012530.


[LongHorizon] Epoch 5: train loss = 0.011880, val loss = 0.011795.


...

[LongHorizon] Epoch 196: train loss = 0.002930, val loss = 0.003622.


[LongHorizon] Epoch 197: train loss = 0.002923, val loss = 0.003624.


[LongHorizon] Epoch 198: train loss = 0.002922, val loss = 0.003690.


[LongHorizon] Epoch 199: train loss = 0.002923, val loss = 0.003668.


[LongHorizon] Epoch 200: train loss = 0.002912, val loss = 0.003605.


## Evaluation

In [11]:
num_eval_eps = 10
nbc_returns = collect_imitator_trajectories(
    env=eval_env,
    policies=nbc_policies,
    num_episodes=num_eval_eps,
    max_steps=num_steps,
    hidden_dims=hidden_dims,
    show_progress=True,
    seed=seed + 90210,
)

len(nbc_returns)

Starting episode 1/10...


  Episode 1 ended at step 2000 (terminated: False, truncated: True).
Starting episode 2/10...


  Episode 2 ended at step 2000 (terminated: False, truncated: True).
Starting episode 3/10...


  Episode 3 ended at step 2000 (terminated: False, truncated: True).
Starting episode 4/10...


  Episode 4 ended at step 2000 (terminated: False, truncated: True).
Starting episode 5/10...


...

  Episode 7 ended at step 2000 (terminated: False, truncated: True).
Starting episode 8/10...


  Episode 8 ended at step 2000 (terminated: False, truncated: True).
Starting episode 9/10...


  Episode 9 ended at step 2000 (terminated: False, truncated: True).
Starting episode 10/10...


  Episode 10 ended at step 2000 (terminated: False, truncated: True).
Finished collecting imitator trajectories.


20000

In [12]:
nbc_episode_rewards = defaultdict(float)
for rec in nbc_returns:
    ep = rec['episode']
    nbc_episode_rewards[ep] += float(rec['reward'])

nbc_rewards = [nbc_episode_rewards[e] for e in range(num_eval_eps)]
sum(nbc_rewards) / num_eval_eps

-835.2918255464767

## Save Model

In [13]:
SAVE_DIR = '/home/et2842/causal/causalrl/models'
os.makedirs(SAVE_DIR, exist_ok=True)
MODEL_PATH = os.path.join(SAVE_DIR, 'nbc_hummed.pt')

checkpoint = {
    "state_dict": nbc_model.state_dict(),
    "slots": nbc_slots,
    "Z_trim": nbc_Z_trim,
    "dims": dims,
    "lookback": lookback,
    "continuous": True,
    "num_actions": train_env.action_space.shape[0],
    "hidden_dim": hidden_size,
    "num_blocks": num_blocks,
    "dropout": dropout,
    "layernorm": True,
    "final_tanh": True,
    "action_bounds_low": eval_env.action_space.low,
    "action_bounds_high": eval_env.action_space.high,
    "input_dim": int(nbc_model.hidden.in_features),
}

torch.save(checkpoint, MODEL_PATH)
print(f'Saved to: {MODEL_PATH}')

Saved to: /home/et2842/causal/causalrl/models/nbc_hummed.pt
